# Lab 02 — Superposition and Measurement

Prepare $R_y(\theta)|0\rangle = \cos\frac{\theta}{2}|0\rangle + \sin\frac{\theta}{2}|1\rangle$,
measure it many times, and compare the measured frequencies with the Born rule
$P(1) = \sin^2\frac{\theta}{2}$. Then measure $|+\rangle$ in two different bases.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(ROOT))

## Parameters

In [ ]:
import numpy as np

THETA = np.pi / 3                         # Ry angle; P(1) = sin^2(THETA/2) = 0.25
SHOTS_LIST = [10, 100, 1_000, 10_000, 100_000]
SEED = 7

In [ ]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

from common.qlab import format_state, save_circuit

backend = AerSimulator(seed_simulator=SEED)

## Step 1 — Prepare the superposition

In [ ]:
qc = QuantumCircuit(1, 1)
qc.ry(THETA, 0)
qc.barrier(label="Superposition")

state = Statevector(qc.remove_final_measurements(inplace=False))
p1_theory = np.sin(THETA / 2) ** 2

print("State:", format_state(*state.data))
print(f"Born rule: P(0) = {1 - p1_theory:.4f}, P(1) = {p1_theory:.4f}")

qc.measure(0, 0)
save_circuit(qc, "circuit.png", scale=1.0);

## Step 2 — Measurement statistics vs number of shots
Each shot is one random outcome. The frequency of `1` converges to $P(1)$, and its
standard error $\sqrt{P(1)(1-P(1))/N}$ shrinks as $1/\sqrt{N}$.

In [ ]:
compiled = transpile(qc, backend)
rows = []

print(f"{'shots':>8} | {'count(1)':>8} | {'P(1) measured':>13} | {'|error|':>8} | {'std. error':>10}")
for shots in SHOTS_LIST:
    counts = backend.run(compiled, shots=shots).result().get_counts()
    p1 = counts.get("1", 0) / shots
    sigma = np.sqrt(p1_theory * (1 - p1_theory) / shots)
    rows.append((shots, p1, sigma))
    print(f"{shots:>8} | {counts.get('1', 0):>8} | {p1:>13.4f} | {abs(p1 - p1_theory):>8.4f} | {sigma:>10.4f}")

plot_histogram(counts, title=f"{SHOTS_LIST[-1]} shots")

In [ ]:
shots_arr, p1_arr, sigma_arr = map(np.array, zip(*rows))

fig, ax = plt.subplots(figsize=(7, 4))
ax.axhline(p1_theory, color="#52514e", linewidth=1.5, linestyle="--", zorder=1)
ax.text(shots_arr[0] * 1.2, p1_theory - 0.008, f"theory {p1_theory:.3f}", va="top", ha="left", color="#52514e")
ax.errorbar(shots_arr, p1_arr, yerr=sigma_arr, color="#2a78d6", linewidth=2,
            marker="o", markersize=8, markeredgecolor="#fcfcfb", markeredgewidth=2,
            capsize=4, zorder=2)

ax.set_xscale("log")
ax.set_xlabel("Number of shots (log scale)", color="#52514e")
ax.set_ylabel("Measured P(1)", color="#52514e")
ax.set_title("Measured P(1) converges to the Born-rule value", color="#0b0b0b", loc="left")
ax.grid(True, color="#e6e5e0", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

fig.savefig("convergence.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 3 — Measurement basis matters
$|+\rangle$ gives random results in the $Z$ basis but a **certain** result in the $X$ basis.
To measure in the $X$ basis, rotate with $H$ first, then measure as usual.

In [ ]:
z_basis = QuantumCircuit(1, 1)
z_basis.h(0)
z_basis.measure(0, 0)

x_basis = QuantumCircuit(1, 1)
x_basis.h(0)
x_basis.h(0)          # rotate X basis -> Z basis
x_basis.measure(0, 0)

shots = SHOTS_LIST[-1]
z_counts = backend.run(transpile(z_basis, backend), shots=shots).result().get_counts()
x_counts = backend.run(transpile(x_basis, backend), shots=shots).result().get_counts()

print("|+> measured in Z basis:", z_counts)
print("|+> measured in X basis:", x_counts)

## Verification

In [ ]:
final_shots, final_p1, final_sigma = rows[-1]
assert abs(final_p1 - p1_theory) < 4 * final_sigma, (final_p1, p1_theory)
assert x_counts == {"0": shots}, x_counts
assert abs(z_counts.get("1", 0) / shots - 0.5) < 4 * np.sqrt(0.25 / shots), z_counts
print("Measured frequencies agree with the Born rule within 4 standard errors.")

In [ ]:
qc.draw(output="mpl", scale=1.0)